## Descripción del dataset: Pima Indians Diabetes

El **Pima Indians Diabetes Dataset** es un conjunto de datos clásico en Machine Learning y bioestadística, recopilado por el *National Institute of Diabetes and Digestive and Kidney Diseases*.  
Su propósito es **predecir la aparición de diabetes tipo 2** en mujeres de origen **pima** (una población indígena del sur de Arizona, EE.UU.), a partir de diversas variables clínicas y demográficas.

### Características principales:
- **Número de registros:** 392 (en esta versión limpia, el original tenía 768).  
- **Número de atributos (features):** 8 variables predictoras + 1 variable objetivo.  
- **Población:** Mujeres de al menos 21 años de edad de la etnia Pima.  
- **Tarea principal:** Clasificación binaria → determinar si una paciente tiene diabetes (`Outcome = 1`) o no (`Outcome = 0`).

### Variables:
1. **Pregnancies** → Número de embarazos.  
2. **Glucose** → Concentración de glucosa en plasma después de 2 horas en una prueba de tolerancia a la glucosa.  
3. **BloodPressure** → Presión arterial diastólica (mm Hg).  
4. **SkinThickness** → Espesor del pliegue cutáneo del tríceps (mm).  
5. **Insulin** → Nivel sérico de insulina (mu U/ml).  
6. **BMI** → Índice de masa corporal (peso en kg / altura² en m²).  
7. **DiabetesPedigreeFunction** → Probabilidad de diabetes basada en antecedentes familiares.  
8. **Age** → Edad en años.  
9. **Outcome** → Variable objetivo:  
   - `0` = No tiene diabetes  
   - `1` = Tiene diabetes  

### Relevancia:
Este dataset es ampliamente utilizado en cursos de **Inteligencia Artificial y Machine Learning** para enseñar:
- Procesamiento y limpieza de datos biomédicos.  
- Métodos de clasificación supervisada (KNN, regresión logística, Random Forest, SVM, redes neuronales, etc.).  
- Importancia de la normalización y estandarización en algoritmos basados en distancias.  

---


## Paso 1: Cargar la base de datos  
Cargamos el CSV en un `DataFrame` de `pandas`. Si tu archivo no se llama exactamente `cleaned_dataset.csv`, ajusta la ruta.

In [5]:
#codigo aqui
import pandas as pd
df = pd.read_csv("/cleaned_dataset.csv")
df.head()

,Pregnancies,Glucose,Blood Pressure,Skin Thickness,Insulin,BMI,Diabetes Pedigree Function,Age,Outcome
0,0,129,110,46,130,67.1,0.319,26,1
1,0,180,78,63,14,59.4,2.420,25,1
2,3,123,100,35,240,57.3,0.880,22,0
3,1,88,30,42,99,55.0,0.496,26,1
4,0,162,76,56,100,53.2,0.759,25,1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Paso 2: Crear subconjuntos con 20 datos de **entrenamiento** y 20 de **testeo**
Seleccionaremos 40 muestras: 20 para entrenar y 20 para evaluar.

In [6]:
#codigo aqui
from sklearn.model_selection import train_test_split

# Separar variables predictoras (X) y variable objetivo (y)
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# Seleccionar 40 muestras del dataset completo
X_40, _, y_40, _ = train_test_split(
    X,
    y,
    train_size=40,
    stratify=y,
    random_state=42
)

# Dividir las 40 muestras en 20 para entrenamiento y 20 para testeo
X_train, X_test, y_train, y_test = train_test_split(
    X_40,
    y_40,
    train_size=20,
    stratify=y_40,
    random_state=42
)

# Verificar tamaños
print("Entrenamiento:", X_train.shape)
print("Testeo:", X_test.shape)


Entrenamiento: (20, 8)
Testeo: (20, 8)


## Paso 3: Implementar la función de distancia euclidiana

**Instrucciones:**
- Escribe una función en Python que reciba dos vectores y calcule la distancia euclidiana entre ellos.
- Utiliza la siguiente fórmula matemática para la distancia euclidiana entre dos vectores $x$ y $y$ de $n$ dimensiones:

$$
d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}
$$

- Prueba tu función con los siguientes dos ejemplos (cada vector corresponde a una fila del dataset):

| Embarazos | Glucosa | Presión Arterial | Grosor Piel | Insulina | IMC  | Función Hereditaria | Edad | Resultado |
|-----------|---------|------------------|-------------|----------|------|---------------------|------|-----------|
|     1     |   106   |        70        |      28     |   135    | 34.2 |        0.142        |  22  |     0     |
|     2     |   102   |        86        |      36     |   120    | 45.5 |        0.127        |  23  |     1     |

- Calcula la distancia euclidiana a mano y luego verifica que el resultado de tu función sea el mismo.
- La función debe imprimir el resultado del cálculo de la distancia euclidiana con los datos presentados.



In [7]:
import numpy as np

def distancia_euclidiana(x, y):
    """
    Calcula la distancia euclidiana entre dos vectores x e y
    """
    return np.sqrt(np.sum((x - y) ** 2))

# Vectores de prueba (sin la variable Resultado)
x = np.array([1, 106, 70, 28, 135, 34.2, 0.142, 22])
y = np.array([2, 102, 86, 36, 120, 45.5, 0.127, 23])

distancia = distancia_euclidiana(x, y)
print("Distancia euclidiana:", distancia)



Distancia euclidiana: 26.280985997484947


## Paso 4: Implementar un clasificador KNN básico

**Instrucciones:**
- Escribe una función que, dado un punto de prueba, calcule la distancia a todos los puntos de entrenamiento utilizando tu función de distancia euclidiana.
- Selecciona los **k = 3** vecinos más cercanos y predice la clase mayoritaria entre ellos.
- Aplica tu función a las 10 muestras de prueba obtenidas previamente, utilizando las 10 muestras de entrenamiento como referencia.
- El script debe imprimir una tabla comparando el valor real de `Resultado` de cada muestra de prueba con el valor predicho por tu algoritmo.
- Considere que las tablas se pueden codificar con un formato similar al que se muestra en el siguiente código:

In [8]:
#codigo aqui
def knn_predict(X_train, y_train, x_test, k=3):
    """
    Predice la clase de un punto de prueba usando KNN manual
    """
    distancias = []

    # Calcular distancia a cada punto de entrenamiento
    for i in range(len(X_train)):
        d = distancia_euclidiana(
            X_train.iloc[i].values,
            x_test.values
        )
        distancias.append((d, y_train.iloc[i]))

    # Ordenar por distancia
    distancias.sort(key=lambda x: x[0])

    # Seleccionar los k vecinos más cercanos
    vecinos = distancias[:k]

    # Votación mayoritaria
    clases = [v[1] for v in vecinos]
    prediccion = max(set(clases), key=clases.count)

    return prediccion


## Paso 5: Usar toda la data con separación 80% entrenamiento / 20% testeo  

### Pasos:
1. Cargar todo el dataset.  
2. Separar variables (X) y etiquetas (y).  
3. Aplicar `train_test_split` con 80% para entrenamiento y 20% para testeo.  
4. Mantener la proporción de clases usando estratificación.  
5. Guardar los conjuntos de datos para usarlos en KNN.  

In [11]:
import pandas as pd

df = pd.read_csv("/cleaned_dataset.csv")

X = df.drop("Outcome", axis=1)  # Variables predictoras
y = df["Outcome"]               # Variable objetivo

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      # 20% para test
    stratify=y,         # Mantener proporción de clases
    random_state=42     # Reproducibilidad
)

print("Entrenamiento:", X_train.shape)
print("Testeo:", X_test.shape)


Entrenamiento: (313, 8)
Testeo: (79, 8)


## Paso 6: Entrenar un KNN con los datos sin escalar (crudos) y calcular accuracy  

### Pasos:
1. Definir el valor de **k = 3** y el metodo **Euclidiano**.  
2. Entrenar el modelo KNN con los datos crudos (sin normalizar/estandarizar).  
3. Predecir las clases del conjunto de test.  
4. Calcular el **accuracy** comparando predicciones con etiquetas reales.  
5. Guardar el resultado para la tabla comparativa.  


In [12]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn_raw = KNeighborsClassifier(
    n_neighbors=3,
    metric="euclidean"
)

knn_raw.fit(X_train, y_train)

y_pred_raw = knn_raw.predict(X_test)

accuracy_raw = accuracy_score(y_test, y_pred_raw)
accuracy_raw

accuracy_raw


0.810126582278481

## Paso 7: Normalizar (Min-Max scaling) y entrenar KNN, luego calcular accuracy  

### Pasos:
1. Aplicar **normalización Min-Max** a los datos de entrenamiento y test.  
2. Entrenar el modelo KNN con los datos normalizados.  
3. Predecir las clases del conjunto de test.  
4. Calcular el **accuracy** del modelo.  
5. Guardar el resultado para la tabla comparativa.  


In [14]:
from sklearn.preprocessing import MinMaxScaler

# Crear el normalizador
scaler_minmax = MinMaxScaler()

# Ajustar SOLO con los datos de entrenamiento
X_train_minmax = scaler_minmax.fit_transform(X_train)

# Aplicar la misma transformación al conjunto de test
X_test_minmax = scaler_minmax.transform(X_test)

from sklearn.neighbors import KNeighborsClassifier

# Crear y entrenar el modelo KNN con los datos normalizados
knn_minmax = KNeighborsClassifier(
    n_neighbors=3,
    metric="euclidean"
)
knn_minmax.fit(X_train_minmax, y_train)

y_pred_minmax = knn_minmax.predict(X_test_minmax)

from sklearn.metrics import accuracy_score

accuracy_minmax = accuracy_score(y_test, y_pred_minmax)
accuracy_minmax

0.7341772151898734

## Paso 9: Estandarizar (Z-score) y entrenar KNN, luego calcular accuracy  

### Pasos:
1. Aplicar **estandarización Z-score** a los datos de entrenamiento y test.  
2. Entrenar el modelo KNN con los datos estandarizados.  
3. Predecir las clases del conjunto de test.  
4. Calcular el **accuracy** del modelo.  
5. Guardar el resultado para la tabla comparativa.  


In [15]:
from sklearn.preprocessing import StandardScaler

# Crear el estandarizador
scaler_standard = StandardScaler()

# Ajustar SOLO con los datos de entrenamiento
X_train_std = scaler_standard.fit_transform(X_train)

# Aplicar la misma transformación al conjunto de test
X_test_std = scaler_standard.transform(X_test)

from sklearn.neighbors import KNeighborsClassifier

knn_std = KNeighborsClassifier(
    n_neighbors=3,
    metric="euclidean"
)

knn_std.fit(X_train_std, y_train)

y_pred_std = knn_std.predict(X_test_std)

from sklearn.metrics import accuracy_score

accuracy_std = accuracy_score(y_test, y_pred_std)
accuracy_std

accuracy_std


0.7468354430379747

## Paso 10/11: Tabla comparativa de accuracies  

### Pasos:
1. Reunir los resultados de accuracy de cada experimento:  
   - KNN sin escalar (80/20).  
   - KNN normalizado (80/20).  
   - KNN estandarizado (80/20).  
2. Crear una tabla con los resultados.  
3. Comparar el desempeño de cada método.  



In [17]:
# Supongamos que ya tenemos las siguientes variables:
# accuracy_raw      -> KNN sin escalar
# accuracy_minmax   -> KNN normalizado
# accuracy_std      -> KNN estandarizado

# Crear un diccionario con los resultados
resultados_accuracies = {
    "Método": [
        "KNN sin escalar",
        "KNN normalizado",
        "KNN estandarizado"
    ],
    "Accuracy": [
        accuracy_raw,
        accuracy_minmax,
        accuracy_std
    ]
}

tabla_accuracies = pd.DataFrame(resultados_accuracies)
tabla_accuracies


,Método,Accuracy
0,KNN sin escalar,0.810127
1,KNN normalizado,0.734177
2,KNN estandarizado,0.746835


---
## Preguntas de reflexión y aplicación



1. ¿Por qué es importante normalizar o estandarizar los datos antes de usar KNN?  



es importante ya que knn se basa en distancia
y la valariables tienen rangos muy distintos y las variables con valores grandes dominarán las distancia y lo que hace Normalizar o estandarizar es qy¿ue pone todas las variables en la misma escala, evitando que unas influyan más que otras y mejorando la precisión de KNN.

2. ¿**Qué diferencias observaste en el accuracy entre los datos crudos, normalizados y estandarizados?**

Sin Escalar: suele dar menor accuracy porque algunas variables dominan la distancia.

Normalizado : mejora el accuracy, ya que todas las variables quedan en el mismo rango [0,1].

Estandarizado : también mejora el accuracy; los valores se centran en media 0 y desviación 1, manteniendo la distribución original.

3. Si aumentamos el valor de **k** (número de vecinos), ¿cómo crees que cambiaría el rendimiento del modelo?  


Aumentar k hace que la predicción dependa de más vecinos, lo que puede suavizar el modelo y reducir el ruido. Si k es demasiado grande, el modelo puede subajustarse y perder capacidad de distinguir clases cercanas y si k es demasiado pequeño, el modelo puede ser muy sensible al ruido (sobreajuste).

4. ¿Qué ventaja tiene implementar KNN manualmente antes de usar scikit-learn?  


Permite entender cómo funciona realmente el algoritmo, paso a paso: cálculo la  distancias, selección de vecinos, votación mayoritaria y Ayuda a diagnosticar errores o interpretar resultados, algo que no es evidente usando solo scikit-learn tabiem Refuerza los conceptos de distancia, escalado y selección de k, que son fundamentales en Machine Learning.

5. ¿Qué limitaciones presenta KNN cuando se aplica a conjuntos de datos grandes o con muchas dimensiones?  

Costos computacionales altos tabien necesitaria  calcular la distancia entre cada punto de prueba y todos los puntos de entrenamiento y  en espacios con muchas dimensiones, las distancias se vuelven menos significativas y KNN puede perder precisión y ser Sensible  al ruido y valores atípicos: un solo dato anómalo puede cambiar la predicción si k es pequeño.



## Rúbrica de evaluación: Práctica KNN

| Criterio | Descripción | Puntaje Máximo |
|----------|-------------|----------------|
| **1. Carga y exploración del dataset** | Carga correcta del archivo CSV, explicación de las variables y verificación de datos. | 15 pts |
| **2. Implementación manual de KNN** | Código propio para calcular distancias euclidianas, selección de vecinos y votación mayoritaria. | 20 pts |
| **3. Predicción individual (ejemplo aleatorio)** | Explicación clara del proceso paso a paso para un ejemplo de test. | 10 pts |
| **4. Uso de scikit-learn (KNN)** | Entrenamiento y evaluación con `train_test_split`, comparación con el método manual. | 15 pts |
| **5. Normalización y estandarización** | Aplicación correcta de Min-Max y Z-score, con cálculo de accuracy en cada caso. | 20 pts |
| **6. Tabla comparativa de accuracies** | Presentación clara de los resultados y comparación entre métodos. | 10 pts |
| **7. Reflexión y preguntas finales** | Respuestas a las preguntas de análisis planteadas (profundidad y claridad). | 10 pts |

**Total: 100 pts**
